# Part 11 (실습) — 중급 프로젝트: 문서 작성 비서 에이전트

> **이 노트북의 목표**
> 중급의 셋(도구·RAG·메모리)을 합쳐 문서 작성 비서를 완성함. 보고서 개요 → 섹션 추가 → PPT 구성으로 이어지는 멀티턴 작업을 돌리고, 근거가 필요한 대목엔 RAG를 쓰며, 마지막에 "검토·승인 절차" 요구로 한계를 마주함.
>
> **사용 모델**: 채팅 `gemini-3-flash`, 임베딩 `gemini-embedding-001` · **선행**: Part 0~10 (중급 졸업)

## 0. 준비

In [ ]:
%pip install -q -U langchain langchain-google-genai langchain-text-splitters

In [ ]:
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0.4)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
print("준비 완료")

## 1. 문서 작업 도구 (Part 7)

문서 작업을 단일 기능 도구들로 나눔. docstring에 "언제 쓰는지" 명시. 도구 내부에서 LLM을 한 번 더 호출(작은 체인).

In [ ]:
def llm_text(prompt: str) -> str:
    return model.invoke(prompt).content.strip()

@tool
def make_outline(topic: str) -> str:
    """주제로 보고서 목차(개요)를 만든다. 보고서·문서 구조가 필요할 때 사용한다."""
    return llm_text(f"'{topic}' 주제의 보고서 목차를 5개 대제목으로, 마크다운 번호 목록으로만 작성해줘.")

@tool
def draft_section(section_title: str, key_points: str) -> str:
    """섹션 제목과 핵심 요점으로 본문 초안을 쓴다. 특정 섹션 글이 필요할 때 사용한다."""
    return llm_text(f"섹션 '{section_title}'의 본문을 핵심요점({key_points})을 반영해 3~4문장으로 써줘.")

@tool
def make_slide_plan(topic: str, num_slides: int) -> str:
    """주제와 장수로 PPT 슬라이드 구성안을 만든다. 발표 자료 구성이 필요할 때 사용한다."""
    return llm_text(f"'{topic}'을 {num_slides}장 PPT로. 각 장의 제목과 핵심 불릿 2개를 마크다운으로만.")

@tool
def summarize_text(text: str) -> str:
    """긴 텍스트를 핵심만 요약한다. 내용을 줄여야 할 때 사용한다."""
    return llm_text(f"다음을 3줄로 요약해줘:\n{text}")

print("문서 작업 도구 4개 준비 완료")

## 2. 참고자료 RAG 도구 (Part 8)

참고 자료를 벡터스토어에 넣고, RAG를 도구로 감싸 "근거가 필요할 때만" 검색하게 함.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

reference = """생성형 AI 업무 도입 참고 자료.
- 2025년 기업의 생성형 AI 도입률은 전년 대비 약 2배 증가했다.
- 가장 큰 효과는 문서 작성·요약 등 반복 업무 시간 단축(평균 30%)이다.
- 주요 리스크는 환각(부정확한 생성), 데이터 보안, 저작권 문제다.
- 성공 도입의 핵심은 명확한 사용 가이드라인과 직원 교육이다.
- 도입 단계: 파일럿 → 부서 확대 → 전사 적용 순으로 점진 도입이 권장된다."""

ref_chunks = RecursiveCharacterTextSplitter(chunk_size=90, chunk_overlap=15).split_text(reference)
ref_store = InMemoryVectorStore.from_texts(ref_chunks, embedding=embeddings)
ref_retriever = ref_store.as_retriever(search_kwargs={"k": 3})

ref_prompt = ChatPromptTemplate.from_template(
    "참고 자료를 근거로만 답해. 없으면 '자료 없음'.\n[자료]\n{context}\n[질문]\n{question}"
)
ref_rag = (
    {"context": ref_retriever | (lambda ds: "\n".join(d.page_content for d in ds)),
     "question": RunnablePassthrough()}
    | ref_prompt | model | StrOutputParser()
)

@tool
def search_reference(query: str) -> str:
    """참고 자료에서 관련 사실·수치·근거를 검색한다.
    보고서에 데이터나 근거가 필요할 때 사용한다."""
    return ref_rag.invoke(query)

print("RAG 도구(search_reference) 준비 완료")

## 3. 비서 에이전트 조립 (Part 9 + 10)

도구 + RAG + system_prompt + 메모리(체크포인터)를 모두 결합함.

In [ ]:
secretary = create_agent(
    model=model,
    tools=[make_outline, draft_section, make_slide_plan, summarize_text, search_reference],
    system_prompt=(
        "너는 유능한 문서 작성 비서야.\n"
        "- 목차는 make_outline, 섹션 글은 draft_section, 발표자료는 make_slide_plan, 요약은 summarize_text를 써.\n"
        "- 수치·사실·근거가 필요하면 반드시 search_reference로 확인하고 인용해. 추측하지 마.\n"
        "- 사용자의 이전 작업물을 기억하고 이어서 다듬어 줘."
    ),
    checkpointer=InMemorySaver(),
)

def work(thread, msg):
    cfg = {"configurable": {"thread_id": thread}}
    r = secretary.invoke({"messages": [("user", msg)]}, config=cfg)
    used = [tc["name"] for m in r["messages"] if getattr(m,"tool_calls",None) for tc in m.tool_calls]
    print(f"🙋 {msg}\n   🔧 쓴 도구: {used or '없음'}\n   🤖 {r['messages'][-1].content}\n")
    return r

print("문서 작성 비서 완성!")

## 4. 멀티턴 문서 작업 — 이어서 다듬기

한 thread에서 개요 → 섹션 추가 → 슬라이드로 이어 작업함. "거기에", "이 보고서를"이 통함(메모리).

In [ ]:
work("report-1", "생성형 AI 업무 도입 보고서 개요를 짜줘");

In [ ]:
work("report-1", "거기에 '도입 리스크' 섹션 초안을 추가해줘");   # "거기" = 앞 개요 (메모리)

In [ ]:
work("report-1", "이 보고서를 5장짜리 PPT 구성안으로 만들어줘");   # "이 보고서" = 지금까지 작업

## 5. 근거가 필요한 작업 — RAG 자동 사용

수치·사실이 필요한 요청엔 에이전트가 search_reference를 골라 자료를 확인함.

In [ ]:
work("report-2", "생성형 AI 도입 효과를 데이터 근거와 함께 한 단락으로 써줘");
# → search_reference로 "시간 30% 단축, 도입률 2배" 등을 가져와 인용

## 6. 졸업 — 그리고 에이전트의 한계

지금 에이전트는 흐름을 모델의 자유 판단에 맡김. "정해진 절차"가 필요한 요구를 던져 봄.

In [ ]:
work("report-3",
     "보고서 초안을 만들되, 반드시 사람 검토를 받고 → 승인되면 발행 → 반려되면 수정 후 재검토하는 "
     "절차를 보장해줘.")
# → 에이전트는 '절차를 설명'하거나 초안만 만들 뿐, "사람 검토에서 멈추고 기다리는" 흐름이나
#   "반려 시 루프로 되돌아가는" 제어를 보장하지 못함.

### 결론 — 무엇이 부족한가
- 지금 에이전트: 흐름을 **모델이 자유롭게** 판단 → 대부분의 작업엔 충분함.
- 하지만 **고정된 단계(초안→검토→승인)**, **반복(반려→수정→재검토)**, **반드시 사람이 개입(승인)**하는 절차는 보장하기 어려움.
- 이건 흐름을 **명시적으로 제어**해야 하는 일 → **LangGraph로 직접 그려야 함**(고급 Part 12~).
- 에이전트는 버려지지 않음 — LangGraph 그래프의 한 노드가 될 수 있음.

## ✏️ 미니 실습

**과제**: 비서에게 "report-1 보고서를 한 문단으로 요약해줘"라고 같은 thread로 요청하기. 앞서 만든 작업물을 기억해 요약하는지 확인.

In [ ]:
# TODO: 직접 작성
# work("report-1", "지금까지 만든 이 보고서를 한 문단으로 요약해줘")

<details><summary>👉 정답</summary>

```python
work("report-1", "지금까지 만든 이 보고서를 한 문단으로 요약해줘")
# → 같은 thread라 앞서 만든 개요·섹션·슬라이드를 기억하고 summarize_text로 요약
```
</details>

## 정리 — 중급 졸업 🎓

| 절 | 한 일 | 쓴 것 |
|---|---|---|
| 1 | 문서 작업 도구 | Part 7 |
| 2 | 참고자료 RAG 도구 | Part 8 |
| 3 | 비서 에이전트 조립 | Part 9 + create_agent |
| 4 | 멀티턴 문서 편집 | Part 10 메모리 |
| 5 | 근거 자동 사용 | RAG 협력 |
| 6 | 한계 마주 | → 고급 예고 |

### 3줄 요약
1. 도구·RAG·메모리를 합쳐 이어 작업하는 문서 작성 비서를 완성함.
2. 도구가 협력하고(검색→작성), 메모리로 멀티턴 편집이 가능함.
3. 흐름을 모델 판단에 맡기는 한계 → 정해진 절차는 LangGraph로(고급).

### ▶ 다음 (Part 12 · 고급 시작)
"왜 LangGraph인가" — 루프·조건 분기·장기 상태·사람 개입을 직접 제어해야 하는 순간을 다룸. 흐름의 운전대를 우리가 쥠.